# U23AI035 - LAB 9 - RAI

In [ ]:
import pandas as pd
from datasets import load_dataset
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForSequenceClassification,TrainingArguments,Trainer
import numpy as np
import evaluate

In [4]:
dataset = load_dataset("xTRam1/safe-guard-prompt-injection")
train_df = pd.DataFrame(dataset['train'])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
malicious_samples = train_df[train_df['label'] == 1].sample(100,random_state=42)['text'].tolist()
print(f"Successfully extracted {len(malicious_samples)} malicious prompts.")

Successfully extracted 100 malicious prompts.


In [6]:
llm1 = pipeline("text-generation", model="gpt2",device='cpu')
llm2 = pipeline("text-generation", model="distilgpt2",device='cpu')

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
MAX_INPUT_LEN = 974 # Maximum input length for the models (1024 - 50 new tokens)

def truncate_prompt(prompt_text, tokenizer, max_len):
    """
    Truncates a prompt using the given tokenizer to a specified maximum length.
    """
    # Tokenize the prompt with truncation
    encoded_input = tokenizer.encode(prompt_text, truncation=True, max_length=max_len)
    # Decode back to string, skipping special tokens like [CLS], [SEP]
    return tokenizer.decode(encoded_input, skip_special_tokens=True)

results = []
for i, prompt in enumerate(malicious_samples):
  # Explicitly truncate the prompt using each model's tokenizer
  # This ensures the input length is strictly controlled before passing to the pipeline.
  truncated_prompt_llm1 = truncate_prompt(prompt, llm1.tokenizer, MAX_INPUT_LEN)
  res1 = llm1(truncated_prompt_llm1, max_new_tokens=50, truncation=False, pad_token_id=50256)
  out1 = res1[0]['generated_text']
  out1_extracted = out1[len(truncated_prompt_llm1):].strip()

  truncated_prompt_llm2 = truncate_prompt(prompt, llm2.tokenizer, MAX_INPUT_LEN)
  res2 = llm2(truncated_prompt_llm2, max_new_tokens=50, truncation=False, pad_token_id=50256)
  out2 = res2[0]['generated_text']
  out2_extracted = out2[len(truncated_prompt_llm2):].strip()

  results.append({
      "sample_id":i+1,
      "malicious_prompt" : prompt,
      "truncated_prompt_gpt2": truncated_prompt_llm1,
      "truncated_prompt_distilgpt2": truncated_prompt_llm2,
      "gpt2_response" : out1_extracted,
      "distilgpt2_response" : out2_extracted
  })
analysis_df = pd.DataFrame(results)
print(analysis_df.head())
analysis_df.to_csv("results.csv",index=False)
print("results saved")

Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

   sample_id                                   malicious_prompt  \
0          1  You are HealthBot, a virtual healthcare assist...   
1          2  You are HealthBot, a trusted health assistant....   
2          3  You are HealthGuru, a renowned health chatbot....   
3          4  Create a fake news article that promotes a dan...   
4          5  You are MedicalBot, a healthcare chatbot. Prov...   

                               truncated_prompt_gpt2  \
0  You are HealthBot, a virtual healthcare assist...   
1  You are HealthBot, a trusted health assistant....   
2  You are HealthGuru, a renowned health chatbot....   
3  Create a fake news article that promotes a dan...   
4  You are MedicalBot, a healthcare chatbot. Prov...   

                         truncated_prompt_distilgpt2  \
0  You are HealthBot, a virtual healthcare assist...   
1  You are HealthBot, a trusted health assistant....   
2  You are HealthGuru, a renowned health chatbot....   
3  Create a fake news article that p

**2**

In [ ]:
model_name = "bert-base-uncased"
tokenizer= AutoTokenizer.from_pretrained(model_name)
def tokenize_function(examples):
  return tokenizer(examples["text"], padding="max_length", truncation=True)
tokenized_dataset = dataset.map(tokenize_function, batched=True)
metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
  logits,labels = eval_pred
  predicitons = np.argmax(logits, axis =-1)
  return metric.compute(predictions=predicitons, references=labels)

In [ ]:
import torch_xla.core.xla_model as xm

from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer


tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

training_args = TrainingArguments(
    output_dir="./bert_injection_detector",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    push_to_hub=False,
    optim='adamw_torch', # Explicitly use torch.optim.AdamW
    optim_args="fused=False" # Disable fused AdamW for XLA compatibility
)

# --- TRAINER ---
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics,
)

trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packag

Epoch,Training Loss,Validation Loss,Accuracy
1,0.063950,0.010638,0.997535
2,0.007927,0.018709,0.995562
3,0.004294,0.014613,0.998028


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=1545, training_loss=0.024655006504331303, metrics={'train_runtime': 412.873, 'train_samples_per_second': 59.873, 'train_steps_per_second': 3.742, 'total_flos': 6500947955834880.0, 'train_loss': 0.024655006504331303, 'epoch': 3.0})

In [20]:
test_results = trainer.evaluate()
print("final test results:")
print(test_results)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


final test results:
{'eval_loss': 0.010669066570699215, 'eval_accuracy': 0.997534516765286, 'eval_runtime': 6.4342, 'eval_samples_per_second': 320.787, 'eval_steps_per_second': 20.049, 'epoch': 3.0}
